流式输出：
invoke() 和 stream()的区别：
invoke()通常会等待整张图执行完成，然后返回最终结果。

stream()会返回一个迭代器，工作流运行期间逐步产出数据块（chunk），你可以边接收边处理，而不必等待最终结果。异步版本则使用 astream() 和 async for

调用stream() 不一定代表一定会逐 token输出，是否逐token生成取决于选择的流模式和模型本身是否支持

流模式：
1. updates：观察每个节点的状态更新，只返回节点的更新
2. values：观察每一步完整的state状态
3. messages：大模型逐 Token 输出，模式输出的是 (message_chunk, metadata) 这样的二元组：message_chunk：模型输出的消息片段。metadata：与这次模型调用相关的元数据，例如节点名称。
4. custom:节点自己发送进度（get_stream_writer()），有时你不需要模型生成 Token，而是希望在执行过程中主动告诉前端：
   - 正在查询数据库。
   - 已找到 100 条数据。
   - 数据处理完成 50%。
   - 正在调用外部 API。
5.同时使用多种流模式
 graph.stream(
    inputs,
    stream_mode=["messages", "updates", "custom"],
    version="v2"
)
这里的version="v2" 是 LangGraph 新版本提供统一的流式事件格式。设置 version="v2" 后，每条事件都具有统一结构：
{
    "type": "updates",
    "ns": (),
    "data": {...}
}
ns是命名空间，子图场景中可用于识别事件来源.


同步流和异步流：stream()、astream()
6.tasks
7.checkpoints










In [1]:
"""
    updates流模式
"""

from typing import TypedDict

from langgraph.graph import StateGraph, START, END


class State(TypedDict):
    number: int


def step_a(state: State):
    return {"number": state["number"] + 1}


def step_b(state: State):
    return {"number": state["number"] * 10}


builder = StateGraph(State)
builder.add_node("step_a", step_a)
builder.add_node("step_b", step_b)

builder.add_edge(START, "step_a")
builder.add_edge("step_a", "step_b")
builder.add_edge("step_b", END)

graph = builder.compile()


for chunk in graph.stream(
    {"number": 1}, # 初始状态
    stream_mode="updates"
):
    print(chunk)

{'step_a': {'number': 2}}
{'step_b': {'number': 20}}


In [2]:
"""
    values流模式
"""

from typing import TypedDict
from langgraph.graph import StateGraph, START, END


class State(TypedDict):
    number: int


def step_a(state: State):
    return {"number": state["number"] + 1}


def step_b(state: State):
    return {"number": state["number"] * 10}


builder = StateGraph(State)
builder.add_node("step_a", step_a)
builder.add_node("step_b", step_b)

builder.add_edge(START, "step_a")
builder.add_edge("step_a", "step_b")
builder.add_edge("step_b", END)

graph = builder.compile()


for chunk in graph.stream(
    {"number": 1}, # 初始状态
    stream_mode="values"
):
    print(chunk)

{'number': 1}
{'number': 2}
{'number': 20}


In [5]:
"""
    messages 模式
"""

import os
from typing import TypedDict

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage
from langgraph.graph import END, START, StateGraph

load_dotenv(override=True)

# ============================================================
# 初始化模型
# ============================================================
api_key = os.getenv("COMPANY_API_KEY")
base_url = os.getenv("COMPANY_BASE_URL")


class State(TypedDict):
    messages: list

model = init_chat_model(
    model="qwen3.7-plus",
    model_provider="openai",
    api_key=api_key,
    base_url=base_url,
)



def call_model(state: State):
    # 虽然节点使用了 model.invoke()，LangGraph 仍可能通过消息流模式捕获模型的 Token 输出。 前提是模型集成支持相应的流式能力；如果模型或调用方式禁用了流式输出，就不能保证逐 Token 输出。
    response = model.invoke(state["messages"])
    return {"messages": [response]}


builder = StateGraph(State)
builder.add_node("call_model", call_model)

builder.add_edge(START, "call_model")
builder.add_edge("call_model", END)

graph = builder.compile()


for message_chunk, metadata in graph.stream(
    {
        "messages": [
            HumanMessage(content="用简单的语言解释 LangGraph")
        ]
    },
    stream_mode="messages"
):
    # print(message_chunk)
    if message_chunk.content:
        print(message_chunk.content, end="", flush=True)

print()




content='' additional_kwargs={} response_metadata={'model_provider': 'openai'} id='lc_run--01a162eb-026d-7993-8dfc-ac05f8a89127' tool_calls=[] invalid_tool_calls=[] tool_call_chunks=[]
content='' additional_kwargs={} response_metadata={'model_provider': 'openai'} id='lc_run--01a162eb-026d-7993-8dfc-ac05f8a89127' tool_calls=[] invalid_tool_calls=[] tool_call_chunks=[]
content='' additional_kwargs={} response_metadata={'model_provider': 'openai'} id='lc_run--01a162eb-026d-7993-8dfc-ac05f8a89127' tool_calls=[] invalid_tool_calls=[] tool_call_chunks=[]
content='' additional_kwargs={} response_metadata={'model_provider': 'openai'} id='lc_run--01a162eb-026d-7993-8dfc-ac05f8a89127' tool_calls=[] invalid_tool_calls=[] tool_call_chunks=[]
content='' additional_kwargs={} response_metadata={'model_provider': 'openai'} id='lc_run--01a162eb-026d-7993-8dfc-ac05f8a89127' tool_calls=[] invalid_tool_calls=[] tool_call_chunks=[]
content='' additional_kwargs={} response_metadata={'model_provider': 'opena

In [6]:
from typing import TypedDict

from langgraph.graph import StateGraph, START, END
from langgraph.config import get_stream_writer


class State(TypedDict):
    result: str


def process(state: State):
    # 使用get_stream_writer()自定义输出流
    writer = get_stream_writer()

    writer({
        "status": "processing",
        "progress": 25
    })

    # 模拟业务处理
    writer({
        "status": "processing",
        "progress": 75
    })

    writer({
        "status": "completed",
        "progress": 100
    })

    return {"result": "处理完成"}


builder = StateGraph(State)
builder.add_node("process", process)

builder.add_edge(START, "process")
builder.add_edge("process", END)

graph = builder.compile()


for chunk in graph.stream(
    {"result": ""},
    stream_mode="custom"
):
    print(chunk)

{'status': 'processing', 'progress': 25}
{'status': 'processing', 'progress': 75}
{'status': 'completed', 'progress': 100}


In [ ]:
from typing import TypedDict

from langgraph.graph import StateGraph, START, END
from langgraph.config import get_stream_writer


class State(TypedDict):
    result: str


def process(state: State):
    # 使用get_stream_writer()自定义输出流
    writer = get_stream_writer()

    writer({
        "status": "processing",
        "progress": 25
    })

    # 模拟业务处理
    writer({
        "status": "processing",
        "progress": 75
    })

    writer({
        "status": "completed",
        "progress": 100
    })

    return {"result": "处理完成"}


builder = StateGraph(State)
builder.add_node("process", process)

builder.add_edge(START, "process")
builder.add_edge("process", END)

graph = builder.compile()


for chunk in graph.stream(
    {"result": ""},
    stream_mode="custom"
):
    print(chunk)